In [5]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from langchain.tools import tool
import requests

In [6]:
from langchain.agents import create_agent

In [7]:
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
os.environ["WEATHERSTACK_API_KEY"] = os.getenv("WEATHERSTACK_API_KEY")

In [8]:
search_tool = TavilySearch(max_results=3)

In [9]:
@tool
def get_weather(city: str) -> str:
    """Fetch current weather information for a city"""

    api_key = os.getenv("WEATHERSTACK_API_KEY")

    if not api_key:
        return "WEATHERSTACK_API_KEY is not set"

    url = (
        f"http://api.weatherstack.com/current"
        f"?access_key={api_key}"
        f"&query={city}"
    )

    response = requests.get(url)
    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}: {data}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [10]:
result = search_tool.invoke("What is the latest news on AI?")
result

{'query': 'What is the latest news on AI?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://guides.library.georgetown.edu/ai/news',
   'title': 'Artificial Intelligence (Generative) Resources: Latest News on AI',
   'content': 'Skip to Main Content\n\n# Artificial Intelligence (Generative) Resources: Latest News on AI\n\n Home\n AI Fundamentals\n AI Tools for Research\n How to Craft Prompts\n Research Resources on AI\n Latest News on AI \n  + AI News from the New York Times, WSJ, and Washington Post (ProQuest)\n  + AI News in Education Periodicals (EBSCO)\n Ethics & AI\n Citing Generative AI\n AI, Authorship, & Copyright\n Campus Resources and Policies\n\n## Publications covering AI\n\n Technology Review This link opens in a new window [...] Loading ...\n\n## AI News in Education Periodicals (EBSCO)\n\nLoading ...\n\n << Previous: Research Resources on AI\n Next: Ethics & AI >>\n\n Last Updated: Jul 17, 2026 5:10 PM\n URL: \n Print Page;)\n\n

In [11]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key = os.environ["GROQ_API_KEY"]
)

In [12]:
response = llm.invoke("What year is it?")
response

AIMessage(content='It’s 2026.', additional_kwargs={'reasoning_content': 'The user asks: "What year is it?" The system message says current date is 2026-09-08. So the answer: 2026. Possibly mention it\'s 2026. The user might want the year. So answer: It\'s 2026.'}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 76, 'total_tokens': 147, 'completion_time': 0.077254176, 'completion_tokens_details': {'reasoning_tokens': 56}, 'prompt_time': 0.004348436, 'prompt_tokens_details': None, 'queue_time': 0.344889494, 'total_time': 0.081602612}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_976a85d17a', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a08048-d2f4-7333-9f42-b1b23de82839-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 76, 'output_tokens': 71, 'total_tokens': 147, 'output_token_details': {'reasoning': 56}})

In [13]:
# tools = []
tools = search_tool,get_weather

In [14]:
# Create Agent
agent = create_agent(
    model = llm,
    tools = tools
)

In [16]:
response = agent.invoke({
    "messages": [
        {"role":"user","content":"Find the current weather in ongole"}
    ]
})
response

{'messages': [HumanMessage(content='Find the current weather in ongole', additional_kwargs={}, response_metadata={}, id='0e42b347-908b-40cb-8b48-5d165135a28a'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants current weather in Ongole. We need to use get_weather function.', 'tool_calls': [{'id': 'fc_031992ba-572d-4477-b6bb-f8ad7c92e546', 'function': {'arguments': '{"city":"Ongole"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 343, 'total_tokens': 385, 'completion_time': 0.043263191, 'completion_tokens_details': {'reasoning_tokens': 17}, 'prompt_time': 0.020621358, 'prompt_tokens_details': None, 'queue_time': 0.563326537, 'total_time': 0.063884549}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_9b8528b477', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0804f-3a8f-7832-90e1-544a3fd703cf-0', t

In [18]:
print(response["messages"][-1].content)

**Current Weather in Ongole**

- **Temperature:** 37 °C  
- **Weather Condition:** Overcast  
- **Humidity:** 35%  

*(Data retrieved via real‑time weather API.)*


In [19]:
for msg in response["messages"]:
    print(msg.content)

Find the current weather in ongole

City: Ongole
Temperature: 37°C
Weather: Overcast 
Humidity: 35%
**Current Weather in Ongole**

- **Temperature:** 37 °C  
- **Weather Condition:** Overcast  
- **Humidity:** 35%  

*(Data retrieved via real‑time weather API.)*
